# LSTM Sequence Model Training
**Project**: Horizon- and Regime-Specific ML for Stock Return Prediction  
**Market**: Indian Equity (NSE) exclusively  
**Run ID**: lstm_sequence_models  

## Architecture
- Small LSTM (64 units) → Dropout(0.3) → LSTM(32) → Dropout(0.2) → Dense(1, sigmoid)
- Input: 20-day lookback window of 44 features
- Early stopping: patience=7 on training-set validation split (10%)
- Scaler: StandardScaler fit on training data ONLY (no test leakage)
- Walk-forward: 6 splits × 2 horizons = 12 combinations

## Cells
1. Auto-detect dataset paths + sanity check  
2. Install / imports  
3. Load data & config  
4. Helper functions (purge, embargo, sequence builder)  
5. LSTM model factory  
6. Walk-forward training loop  
7. Results aggregation & pivot  
8. Save artifacts  
9. ZIP export

In [ ]:
# --- CELL 1: AUTO-DETECT PATHS ----------------------------------------
import os

def find_project_dirs(input_root='/kaggle/input'):
    config_dir = data_dir = None
    for root, dirs, files in os.walk(input_root):
        depth = root.replace(input_root, '').count(os.sep)
        if depth > 5:
            dirs[:] = []
            continue
        if config_dir is None and 'configs' in dirs:
            candidate = os.path.join(root, 'configs')
            if os.path.isfile(os.path.join(candidate, 'experiment.yaml')):
                config_dir = candidate
        if data_dir is None and 'processed' in dirs:
            candidate = os.path.join(root, 'processed')
            if os.path.isfile(os.path.join(candidate, 'features_targets.csv')):
                data_dir = candidate
        if config_dir and data_dir:
            break
    return config_dir, data_dir

CONFIG_DIR, DATA_DIR = find_project_dirs()

if CONFIG_DIR is None or DATA_DIR is None:
    print('ERROR: could not find required dirs. Actual structure:')
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth > 4: dirs[:] = []; continue
        print('  ' * depth + os.path.basename(root) + '/')
        for f in files[:3]: print('  ' * depth + '  ' + f)
    raise RuntimeError('Dataset not found — add nse-ml-project via Add Data panel.')

OUTPUT_DIR  = '/kaggle/working/lstm_sequence_output'
RUN_ID      = 'lstm_sequence_models'
RANDOM_SEED = 42
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Paths OK')
print(f'  CONFIG_DIR : {CONFIG_DIR}')
print(f'  DATA_DIR   : {DATA_DIR}')
required = ['experiment.yaml','models.yaml'], ['features_targets.csv','feature_list.txt']
for fdir, flist in zip([CONFIG_DIR, DATA_DIR], required):
    for fname in flist:
        path = os.path.join(fdir, fname)
        print(f'  [{"OK" if os.path.isfile(path) else "MISSING"}] {fname}')


In [ ]:
# --- CELL 2: INSTALLS & IMPORTS ----------------------------------------
!pip install tensorflow pyyaml scikit-learn pandas numpy --quiet

import warnings, json, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yaml

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, log_loss
)

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')


In [ ]:
# --- CELL 3: LOAD DATA & CONFIG ----------------------------------------
with open(f'{CONFIG_DIR}/experiment.yaml') as f:
    exp_cfg = yaml.safe_load(f)
with open(f'{CONFIG_DIR}/models.yaml') as f:
    mdl_cfg = yaml.safe_load(f)

SPLITS_CFG = exp_cfg['walk_forward']['splits']
HORIZONS   = exp_cfg['horizons']        # [1, 5]
BENCHMARK  = exp_cfg['data']['benchmark']

df = pd.read_csv(f'{DATA_DIR}/features_targets.csv', parse_dates=['Date'])
with open(f'{DATA_DIR}/feature_list.txt') as f:
    FEATURE_COLS = [l.strip() for l in f if l.strip()]

df_stocks = df[df['Ticker'] != BENCHMARK].copy().reset_index(drop=True)
print(f'Stocks df: {df_stocks.shape}  |  Features: {len(FEATURE_COLS)}')
print(f'Date range: {df_stocks["Date"].min().date()} -> {df_stocks["Date"].max().date()}')
print(f'Splits: {len(SPLITS_CFG)}  |  Horizons: {HORIZONS}')


In [ ]:
# --- CELL 4: HELPERS (purge, embargo, sequence builder) ----------------
SEQ_LEN      = 20    # lookback window — 20 trading days (pre-specified)
EMBARGO_DAYS = 5
REGIME_NAMES = {0: 'Low', 1: 'Medium', 2: 'High'}

ALL_DATES = np.sort(df_stocks['Date'].values.astype('datetime64[ns]'))

def get_purge_cutoff(test_start, horizon):
    ts_np  = np.datetime64(test_start, 'ns')
    ts_idx = np.searchsorted(ALL_DATES, ts_np, side='left')
    return pd.Timestamp(ALL_DATES[max(ts_idx - horizon, 0)])

def get_embargo_cutoff(purge_cutoff, embargo_days):
    pc_np  = np.datetime64(purge_cutoff, 'ns')
    pc_idx = np.searchsorted(ALL_DATES, pc_np, side='right')
    return pd.Timestamp(ALL_DATES[min(pc_idx + embargo_days - 1, len(ALL_DATES)-1)])

def clean(X):
    return np.where(np.isfinite(X), X, 0.0)

def build_sequences(X, y, seq_len=SEQ_LEN):
    """
    Build (n_samples, seq_len, n_features) sequences.
    Each sample at position i uses features from [i-seq_len, i).
    The label for position i is y[i].
    Only purely backward-looking — no future data ever used.
    """
    n = len(X)
    xs, ys = [], []
    for i in range(seq_len, n):          # start at seq_len so window is full
        xs.append(X[i - seq_len : i])    # window: rows i-seq_len to i-1
        ys.append(y[i])                  # label: row i
    return np.array(xs, dtype=np.float32), np.array(ys, dtype=np.float32)

def compute_metrics(y_true, y_pred, y_prob, prefix=''):
    n = len(y_true)
    if n == 0:
        return {}
    m = {
        f'{prefix}n':         n,
        f'{prefix}accuracy':  round(accuracy_score(y_true, y_pred), 4),
        f'{prefix}f1':        round(f1_score(y_true, y_pred, zero_division=0), 4),
        f'{prefix}precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        f'{prefix}recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
    }
    if len(np.unique(y_true)) == 2:
        m[f'{prefix}roc_auc']  = round(roc_auc_score(y_true, y_prob), 4)
        m[f'{prefix}log_loss'] = round(log_loss(y_true, y_prob), 4)
    else:
        m[f'{prefix}roc_auc'] = m[f'{prefix}log_loss'] = None
    return m

print(f'Sequence length: {SEQ_LEN} days  |  Embargo: {EMBARGO_DAYS} days')
print('Helpers defined.')


In [ ]:
# --- CELL 5: LSTM MODEL FACTORY ----------------------------------------
def build_lstm(n_features, seq_len=SEQ_LEN, seed=42):
    """
    Small LSTM for binary classification (up/down).
    Architecture (pre-specified, not tuned on test data):
      LSTM(64, return_sequences=True) -> Dropout(0.3)
      LSTM(32)                        -> Dropout(0.2)
      Dense(16, relu)                 -> Dense(1, sigmoid)

    Regularisation choices:
    - Dropout on both LSTM layers
    - Small hidden size (64/32) relative to input
    - Early stopping (patience=7) on validation loss
    - No look-ahead: sequences built entirely backward
    """
    tf.random.set_seed(seed)
    inp = keras.Input(shape=(seq_len, n_features))
    x = layers.LSTM(64, return_sequences=True,
                    dropout=0.3, recurrent_dropout=0.0)(inp)
    x = layers.LSTM(32, return_sequences=False,
                    dropout=0.2, recurrent_dropout=0.0)(x)
    x = layers.Dense(16, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

# Quick architecture summary (using split_1 feature count)
demo = build_lstm(n_features=len(FEATURE_COLS))
demo.summary()


In [ ]:
# --- CELL 6: WALK-FORWARD LSTM TRAINING LOOP ---------------------------
# LEAKAGE SAFEGUARDS:
#  1. Scaler fitted on training data only
#  2. Sequences built AFTER scaling — window is [t-20, t-1], label at t
#  3. Purge + embargo applied before any model sees test dates
#  4. Validation split (10%) carved from the END of training (chronological)
#  5. Early stopping monitors validation loss — never test loss

MAX_EPOCHS  = 50
PATIENCE    = 7
BATCH_SIZE  = 256
VAL_FRAC    = 0.10    # 10% of training tail for early stopping

all_results = []
all_preds   = []
total_combos = len(SPLITS_CFG) * len(HORIZONS)
combo_idx = 0

for split_cfg in SPLITS_CFG:
    label       = split_cfg['label']
    train_start = pd.Timestamp(split_cfg['train_start'])
    train_end   = pd.Timestamp(split_cfg['train_end'])
    test_start  = pd.Timestamp(split_cfg['test_start'])
    test_end    = pd.Timestamp(split_cfg['test_end'])
    regime_col  = f'regime_{label}'

    for horizon in HORIZONS:
        combo_idx += 1
        t0 = time.time()
        target_col = f'fwd_label_{horizon}d'
        return_col = f'fwd_return_{horizon}d'

        print(f'\n[{combo_idx}/{total_combos}] {label}  h={horizon}d', flush=True)

        # ── Purge + embargo ──────────────────────────────────────────────
        purge_cutoff   = get_purge_cutoff(test_start, horizon)
        embargo_cutoff = get_embargo_cutoff(purge_cutoff, EMBARGO_DAYS)

        train_mask = (
            (df_stocks['Date'] >= train_start) &
            (df_stocks['Date'] <= purge_cutoff) &
            df_stocks[target_col].notna()
        )
        test_mask = (
            (df_stocks['Date'] >= test_start) &
            (df_stocks['Date'] <= test_end) &
            df_stocks[target_col].notna()
        )

        df_train = df_stocks[train_mask].copy()
        df_test  = df_stocks[test_mask].copy()

        print(f'  Train: {len(df_train):,}  |  Test: {len(df_test):,}', flush=True)

        if len(df_train) < SEQ_LEN * 5 or len(df_test) < SEQ_LEN:
            print('  SKIP: insufficient data', flush=True)
            continue

        # ── Scale — fit on training only ──────────────────────────────────
        X_train_raw = clean(df_train[FEATURE_COLS].values)
        X_test_raw  = clean(df_test[FEATURE_COLS].values)
        y_train     = df_train[target_col].values.astype(np.float32)
        y_test      = df_test[target_col].values.astype(np.float32)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)
        X_test_scaled  = scaler.transform(X_test_raw)

        # ── Build sequences ──────────────────────────────────────────────
        # IMPORTANT: sequences are built PER-TICKER then concatenated
        # This prevents a sequence from spanning two different stocks
        # (which would be a data contamination error)
        train_tickers = df_train['Ticker'].values
        test_tickers  = df_test['Ticker'].values
        train_dates   = df_train['Date'].values
        test_dates    = df_test['Date'].values

        Xs_train, ys_train, ts_train_dates, ts_train_tickers = [], [], [], []
        Xs_test,  ys_test,  ts_test_dates,  ts_test_tickers  = [], [], [], []
        test_regime_labels = []
        test_return_vals   = []
        test_label_vals    = []

        for ticker in df_stocks['Ticker'].unique():
            # Training sequences for this ticker
            tr_mask = train_tickers == ticker
            if tr_mask.sum() > SEQ_LEN:
                xs, ys = build_sequences(X_train_scaled[tr_mask], y_train[tr_mask])
                Xs_train.append(xs)
                ys_train.append(ys)
                ts_train_dates.extend(train_dates[tr_mask][SEQ_LEN:])
                ts_train_tickers.extend([ticker] * len(xs))

            # Test sequences: prepend last SEQ_LEN rows from training for warm-up
            # so the FIRST test prediction has a full window
            te_mask = test_tickers == ticker
            tr_tail = X_train_scaled[tr_mask][-SEQ_LEN:]   # last SEQ_LEN training rows
            if te_mask.sum() > 0 and len(tr_tail) == SEQ_LEN:
                X_te_full = np.vstack([tr_tail, X_test_scaled[te_mask]])
                y_te_full = np.concatenate([np.zeros(SEQ_LEN), y_test[te_mask]])
                xs_te, ys_te = build_sequences(X_te_full, y_te_full)
                Xs_test.append(xs_te)
                ys_test.append(ys_te)
                ts_test_dates.extend(test_dates[te_mask])
                ts_test_tickers.extend([ticker] * len(xs_te))
                # Metadata for predictions output
                regime_vals = df_test.loc[te_mask, regime_col].values if regime_col in df_test.columns else np.full(te_mask.sum(), -1)
                test_regime_labels.extend(regime_vals)
                test_return_vals.extend(df_test.loc[te_mask, return_col].values)
                test_label_vals.extend(y_test[te_mask])

        if not Xs_train or not Xs_test:
            print('  SKIP: sequence building produced empty arrays', flush=True)
            continue

        X_seq_train = np.concatenate(Xs_train)
        y_seq_train = np.concatenate(ys_train)
        X_seq_test  = np.concatenate(Xs_test)
        y_seq_test  = np.concatenate(ys_test)

        print(f'  Seq train: {X_seq_train.shape}  |  Seq test: {X_seq_test.shape}', flush=True)

        # ── Chronological validation split (from training tail) ──────────
        # NEVER use test data for early stopping — carve from training only
        n_val   = max(int(len(X_seq_train) * VAL_FRAC), SEQ_LEN)
        n_train = len(X_seq_train) - n_val
        X_tr, X_val = X_seq_train[:n_train], X_seq_train[n_train:]
        y_tr, y_val = y_seq_train[:n_train], y_seq_train[n_train:]

        # ── Build & train model ──────────────────────────────────────────
        keras.backend.clear_session()
        model = build_lstm(n_features=len(FEATURE_COLS), seq_len=SEQ_LEN, seed=RANDOM_SEED)

        cb_list = [
            callbacks.EarlyStopping(
                monitor='val_loss', patience=PATIENCE,
                restore_best_weights=True, verbose=0
            ),
            callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=0
            ),
        ]

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=cb_list,
            verbose=0,
            shuffle=False,   # DO NOT shuffle — time series order must be preserved
        )

        epochs_run = len(history.history['loss'])
        best_val_loss = min(history.history['val_loss'])
        print(f'  Epochs: {epochs_run}  |  Best val_loss: {best_val_loss:.4f}', flush=True)

        # ── Evaluate on test ─────────────────────────────────────────────
        y_prob = model.predict(X_seq_test, verbose=0).flatten()
        y_pred = (y_prob >= 0.5).astype(int)
        y_true = y_seq_test.astype(int)

        overall = compute_metrics(y_true, y_pred, y_prob)
        row = {
            'run_id': RUN_ID, 'split': label, 'horizon': horizon,
            'model': 'LSTM',
            'train_rows': len(df_train), 'test_rows': len(df_test),
            'train_start': str(train_start.date()), 'train_end': str(train_end.date()),
            'test_start': str(test_start.date()), 'test_end': str(test_end.date()),
            'seq_len': SEQ_LEN, 'epochs_run': epochs_run,
            'best_val_loss': round(best_val_loss, 4),
            **overall,
        }

        # ── Per-regime metrics ────────────────────────────────────────────
        test_regime_arr = np.array(test_regime_labels)
        for regime_id, regime_name in REGIME_NAMES.items():
            mask = test_regime_arr == regime_id
            if mask.sum() < 5: continue
            regime_metrics = compute_metrics(
                y_true[mask], y_pred[mask], y_prob[mask],
                prefix=f'regime_{regime_name.lower()}_'
            )
            row.update(regime_metrics)

        all_results.append(row)

        # ── Row-level predictions ─────────────────────────────────────────
        n_preds = len(y_true)
        pred_df = pd.DataFrame({
            'Date':      ts_test_dates[:n_preds],
            'Ticker':    ts_test_tickers[:n_preds],
            'split':     label,
            'horizon':   horizon,
            'model':     'LSTM',
            'y_true':    y_true,
            'y_pred':    y_pred,
            'y_prob':    y_prob,
            'regime':    test_regime_arr[:n_preds],
            return_col:  test_return_vals[:n_preds],
        })
        all_preds.append(pred_df)

        acc = overall.get('accuracy', 'n/a')
        auc = overall.get('roc_auc', 'n/a')
        print(f'  LSTM  acc={acc}  auc={auc}  ({time.time()-t0:.0f}s)', flush=True)

        # Free GPU memory between splits
        keras.backend.clear_session()

print('\n=== Walk-forward LSTM loop complete ===')


In [ ]:
# --- CELL 7: RESULTS AGGREGATION ----------------------------------------
results_df = pd.DataFrame(all_results)
preds_df   = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()

print('Results shape:', results_df.shape)
print()
print(results_df[['split','horizon','accuracy','roc_auc','f1','epochs_run']].to_string(index=False))

print()
print('Mean accuracy by horizon:')
print(results_df.groupby('horizon')['accuracy'].mean().round(4))

print()
print('Mean ROC-AUC by horizon:')
print(results_df.groupby('horizon')['roc_auc'].mean().round(4))

# Per-regime
for regime in ['low','medium','high']:
    col = f'regime_{regime}_accuracy'
    if col in results_df.columns:
        val = results_df.groupby('horizon')[col].mean().round(4)
        print(f'\n{regime.title()} vol accuracy by horizon:')
        print(val)

print()
print('Best val_loss per split-horizon:')
print(results_df[['split','horizon','best_val_loss','epochs_run']].to_string(index=False))


In [ ]:
# --- CELL 8: SAVE ARTIFACTS ---------------------------------------------
results_path = f'{OUTPUT_DIR}/lstm_results_all_splits.csv'
preds_path   = f'{OUTPUT_DIR}/lstm_predictions_all_splits.csv'

results_df.to_csv(results_path, index=False)
if not preds_df.empty:
    preds_df.to_csv(preds_path, index=False)

print(f'Saved: {results_path}')
print(f'Saved: {preds_path}')

# Checkpoint summary
print()
print('=' * 60)
print('CHECKPOINT SUMMARY — LSTM Sequence Model Training')
print('=' * 60)
print(f'Run ID      : {RUN_ID}')
print(f'Architecture: LSTM(64) -> LSTM(32) -> Dense(16) -> Dense(1)')
print(f'Seq length  : {SEQ_LEN} days')
print(f'Max epochs  : {MAX_EPOCHS}  |  Patience: {PATIENCE}')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Val fraction: {VAL_FRAC} (chronological tail of training)')
print(f'Results     : {len(results_df)} rows')
if not results_df.empty:
    print(f'Mean acc 1d : {results_df[results_df["horizon"]==1]["accuracy"].mean():.4f}')
    print(f'Mean acc 5d : {results_df[results_df["horizon"]==5]["accuracy"].mean():.4f}')
    print(f'Mean AUC 1d : {results_df[results_df["horizon"]==1]["roc_auc"].mean():.4f}')
    print(f'Mean AUC 5d : {results_df[results_df["horizon"]==5]["roc_auc"].mean():.4f}')
print('=' * 60)


In [ ]:
# --- CELL 9: FINAL ZIP EXPORT -------------------------------------------
import shutil

zip_base = f'/kaggle/working/{RUN_ID}_artifacts'
shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
final_zip = zip_base + '.zip'
size_mb = os.path.getsize(final_zip) / 1e6
print(f'ZIP created: {final_zip}  ({size_mb:.2f} MB)')
print()
print('Next steps:')
print('  1. Download ZIP from Kaggle Output tab')
print(f'  2. Extract to  artifacts/kaggle/{RUN_ID}/')
print('  3. Share lstm_results_all_splits.csv — agent will update RESULTS.md')
print('  4. Proceed to portfolio backtest locally')
